# Data Loading and Profiling

**Author:** Darshan Ramesh  
**Date:** 29 June 2026  
**Dissertation:** AI-Driven Cashflow Forecasting for Data-Scarce SMEs

## Objective
Load 3 years of SME sales and purchase invoice data, validate structure, and 
produce a cleaned weekly net cashflow series for downstream modelling.

## Data sources
6 Excel files, 3 financial years (2023-24, 2024-25, 2025-26).
Each year has both a SALES file and a PURCHASE file.

### Loading rule established
- **Sales Register** → loaded from each year's SALES file
- **Purchase Register** → loaded from each year's PURCHASE file
- Other sheets (TR ENCLAVE, cross-file duplicates) ignored

## Structural issues resolved
- Excel files have 3 title rows above the real header → use `header=3`
- Sheet names are `Sales Register` and `Purchase Register` in every file
- Year 1 purchase data is read from the PURCHASE file (clarified by 
  domain owner)

## Final dataset
- **Sales:** 547 invoices across 3 years
- **Purchases:** 1,742 invoices across 3 years
- Critical columns confirmed: `Date` (datetime) and `Gross Total` (float64)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("All imports successful")
print(f"pandas version:     {pd.__version__}")
print(f"numpy version:      {np.__version__}")
print(f"matplotlib version: {plt.matplotlib.__version__}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

import os
FIG_DIR = "../reports/figures"
os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
# File paths
RAW_DATA_PATH = "../data/raw/"

FILES = {
    "sales": [
        "1 CRG 2023-24 SALES.xlsx",
        "3 CRG 2024-25 SALES.xlsx",
        "5 CRG 2025-26 SALES.xlsx",
    ],
    "purchases": [
        "2 CRG 2023-24 PURCHASE.xlsx",
        "4 CRG 2024-25 PUR.xlsx",
        "6 CRG 2025-26 PUR.xlsx",
    ]
}

# Verify all files exist before loading anything
print("Checking files:")
all_found = True
for category, file_list in FILES.items():
    for f in file_list:
        full_path = os.path.join(RAW_DATA_PATH, f)
        exists = os.path.exists(full_path)
        status = "FOUND  " if exists else "MISSING"
        print(f"  [{status}]  {f}")
        if not exists:
            all_found = False

print()
if all_found:
    print("All 6 files found. Ready to load.")
else:
    print("Fix the MISSING files before continuing.")

In [ ]:
# Inspect all sheet names across all 6 files
print("Sheet names in each file:")
print()

all_files = FILES['sales'] + FILES['purchases']

for f in all_files:
    path = os.path.join(RAW_DATA_PATH, f)
    xl = pd.ExcelFile(path, engine='openpyxl')
    print(f"  {f}")
    for i, sheet in enumerate(xl.sheet_names):
        print(f"    Sheet {i}: '{sheet}'")
    print()

In [ ]:
SALES_FILES = [
    "1 CRG 2023-24 SALES.xlsx",
    "3 CRG 2024-25 SALES.xlsx",
    "5 CRG 2025-26 SALES.xlsx",
]
PURCHASE_FILES = [
    "2 CRG 2023-24 PURCHASE.xlsx",
    "4 CRG 2024-25 PUR.xlsx",
    "6 CRG 2025-26 PUR.xlsx",
]
SALES_SHEET    = 'Sales Register'
PURCHASE_SHEET = 'Purchase Register'

def load_sheet(filepath, sheet_name):
    """Load a sheet — headers are on row 4 (index 3) due to title rows above."""
    try:
        df = pd.read_excel(
            filepath,
            sheet_name=sheet_name,
            header=3,             # <-- the real header is row index 3 (row 4 in Excel)
            engine='openpyxl'
        )
        df['source_file'] = os.path.basename(filepath)
        return df
    except Exception as e:
        print(f"  ERROR — {os.path.basename(filepath)} / {sheet_name}: {e}")
        return None


# Re-load Sales with correct header
print("Loading Sales Register from SALES files...")
sales_frames = []
for f in SALES_FILES:
    path = os.path.join(RAW_DATA_PATH, f)
    df = load_sheet(path, SALES_SHEET)
    if df is not None:
        sales_frames.append(df)
        print(f"  {f} → {df.shape[0]:,} rows")

sales_raw = pd.concat(sales_frames, ignore_index=True)
print(f"\nCombined sales: {sales_raw.shape[0]:,} rows x {sales_raw.shape[1]} cols")
print(f"Columns: {list(sales_raw.columns)}")
print()

# Re-load Purchases with correct header
print("Loading Purchase Register from PURCHASE files...")
purchase_frames = []
for f in PURCHASE_FILES:
    path = os.path.join(RAW_DATA_PATH, f)
    df = load_sheet(path, PURCHASE_SHEET)
    if df is not None:
        purchase_frames.append(df)
        print(f"  {f} → {df.shape[0]:,} rows")

purchase_raw = pd.concat(purchase_frames, ignore_index=True)
print(f"\nCombined purchases: {purchase_raw.shape[0]:,} rows x {purchase_raw.shape[1]} cols")
print(f"Columns: {list(purchase_raw.columns)}")

In [ ]:
def profile_dataframe(df, name):
    print(f"\n{'='*60}")
    print(f"PROFILE: {name}")
    print(f"{'='*60}")
    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
    profile = pd.DataFrame({
        'dtype'      : df.dtypes,
        'null_count' : df.isnull().sum(),
        'null_pct'   : (df.isnull().sum() / len(df) * 100).round(1),
        'unique_vals': df.nunique()
    })
    display(profile)
    
profile_dataframe(sales_raw, "Sales — Real Columns")
profile_dataframe(purchase_raw, "Purchases — Real Columns")

In [ ]:
print("Null counts after reload:")
print(f"  Sales Date nulls:     {sales_raw['Date'].isnull().sum()}")
print(f"  Purchase Date nulls:  {purchase_raw['Date'].isnull().sum()}")

print(f"\nDate dtype:")
print(f"  Sales:    {sales_raw['Date'].dtype}")
print(f"  Purchase: {purchase_raw['Date'].dtype}")

# Also check for empty strings or whitespace masquerading as values
print(f"\nNon-null but suspicious values (sales):")
print(sales_raw['Date'].apply(lambda x: type(x).__name__).value_counts())

In [ ]:
print("=" * 60)
print("SALES — rows with null Date")
print("=" * 60)
sales_null_dates = sales_raw[sales_raw['Date'].isnull()]
print(f"Found: {len(sales_null_dates)} rows")
display(sales_null_dates[['Date', 'Particulars', 'Buyer', 'Voucher No.', 'Gross Total', 'source_file']])

print()
print("=" * 60)
print("PURCHASES — rows with null Date")
print("=" * 60)
purchase_null_dates = purchase_raw[purchase_raw['Date'].isnull()]
print(f"Found: {len(purchase_null_dates)} rows")
display(purchase_null_dates[['Date', 'Particulars', 'Supplier', 'Voucher No.', 'Gross Total', 'source_file']])

In [ ]:
# Verify Grand Total rows match sum of actual transactions per year
print("=" * 70)
print("VERIFY: Does each Grand Total equal the sum of its file's transactions?")
print("=" * 70)

for source_file, group in sales_raw.groupby('source_file'):
    grand_total_row = group[group['Particulars'] == 'Grand Total']
    actual_rows    = group[group['Particulars'] != 'Grand Total']
    
    if len(grand_total_row) > 0:
        stated_total = grand_total_row['Gross Total'].iloc[0]
        computed_sum = actual_rows['Gross Total'].sum()
        diff = stated_total - computed_sum
        match = "MATCH" if abs(diff) < 1 else "MISMATCH"
        print(f"\n{source_file}")
        print(f"  Stated Grand Total:     {stated_total:>16,.2f}")
        print(f"  Sum of actual rows:     {computed_sum:>16,.2f}")
        print(f"  Difference:             {diff:>16,.2f}  [{match}]")

print("\n" + "=" * 70)
print("Same check for PURCHASES:")
print("=" * 70)

for source_file, group in purchase_raw.groupby('source_file'):
    grand_total_row = group[group['Particulars'] == 'Grand Total']
    actual_rows    = group[group['Particulars'] != 'Grand Total']
    
    if len(grand_total_row) > 0:
        stated_total = grand_total_row['Gross Total'].iloc[0]
        computed_sum = actual_rows['Gross Total'].sum()
        diff = stated_total - computed_sum
        match = "MATCH" if abs(diff) < 1 else "MISMATCH"
        print(f"\n{source_file}")
        print(f"  Stated Grand Total:     {stated_total:>16,.2f}")
        print(f"  Sum of actual rows:     {computed_sum:>16,.2f}")
        print(f"  Difference:             {diff:>16,.2f}  [{match}]")

In [ ]:
# Exclude summary rows from date analysis
sales_valid    = sales_raw.dropna(subset=['Date'])
purchase_valid = purchase_raw.dropna(subset=['Date'])

print("=" * 60)
print("DATE RANGE SUMMARY")
print("=" * 60)
print(f"\nSales:")
print(f"  First transaction:  {sales_valid['Date'].min().date()}")
print(f"  Last transaction:   {sales_valid['Date'].max().date()}")
print(f"  Span:               {(sales_valid['Date'].max() - sales_valid['Date'].min()).days} days")

print(f"\nPurchases:")
print(f"  First transaction:  {purchase_valid['Date'].min().date()}")
print(f"  Last transaction:   {purchase_valid['Date'].max().date()}")
print(f"  Span:               {(purchase_valid['Date'].max() - purchase_valid['Date'].min()).days} days")

# Check for out-of-range dates (before Apr 2023 or after Mar 2026 — Indian FY boundaries)
print("\n" + "=" * 60)
print("OUT-OF-RANGE CHECK (expected: 2023-04-01 to 2026-03-31)")
print("=" * 60)

lower_bound = pd.Timestamp('2023-04-01')
upper_bound = pd.Timestamp('2026-03-31')

sales_ooo    = sales_valid[(sales_valid['Date'] < lower_bound) | (sales_valid['Date'] > upper_bound)]
purchase_ooo = purchase_valid[(purchase_valid['Date'] < lower_bound) | (purchase_valid['Date'] > upper_bound)]

print(f"Sales out-of-range rows:     {len(sales_ooo)}")
print(f"Purchase out-of-range rows:  {len(purchase_ooo)}")

if len(sales_ooo) > 0:
    display(sales_ooo[['Date', 'Particulars', 'Voucher No.', 'Gross Total', 'source_file']].head(10))
if len(purchase_ooo) > 0:
    display(purchase_ooo[['Date', 'Particulars', 'Voucher No.', 'Gross Total', 'source_file']].head(10))

# Gap check — how many days between consecutive transaction dates?
print("\n" + "=" * 60)
print("TRANSACTION GAPS (unique dates only)")
print("=" * 60)

for name, df in [('Sales', sales_valid), ('Purchases', purchase_valid)]:
    dates = df['Date'].sort_values().unique()
    gaps = pd.Series(dates).diff().dt.days.dropna()
    print(f"\n{name}:")
    print(f"  Unique transaction dates:  {len(dates)}")
    print(f"  Median gap between dates:  {gaps.median():.1f} days")
    print(f"  Max gap between dates:     {gaps.max():.0f} days")
    print(f"  Gaps > 14 days:            {(gaps > 14).sum()}")

In [ ]:
def analyse_gross_total(df, name):
    """Descriptive statistics and quality flags for Gross Total."""
    print("=" * 60)
    print(f"{name.upper()} — Gross Total analysis")
    print("=" * 60)

    # Exclude summary rows for a clean analysis
    df_valid = df.dropna(subset=['Date'])
    gt = df_valid['Gross Total']

    print(f"\nTransactions analysed:       {len(gt):,}")
    print(f"Nulls in Gross Total:        {gt.isnull().sum()}")

    print(f"\nDescriptive statistics:")
    print(gt.describe().apply('{:,.2f}'.format))

    print(f"\nQuality flags:")
    print(f"  Zero-value transactions:   {(gt == 0).sum()}")
    print(f"  Negative transactions:     {(gt < 0).sum()}")
    print(f"  Positive transactions:     {(gt > 0).sum()}")

    # Distribution shape
    print(f"\nDistribution shape:")
    print(f"  Skewness:                  {gt.skew():.2f}")
    print(f"  Kurtosis:                  {gt.kurtosis():.2f}")

    # Percentile view — better than mean/std for skewed financial data
    print(f"\nPercentile view:")
    for p in [50, 75, 90, 95, 99, 99.5]:
        print(f"  {p}th percentile:          {gt.quantile(p/100):>16,.2f}")

analyse_gross_total(sales_raw, "sales")
print()
analyse_gross_total(purchase_raw, "purchases")

In [ ]:
sales_null_gt = sales_raw[sales_raw['Gross Total'].isnull() & sales_raw['Date'].notnull()]
purchase_null_gt = purchase_raw[purchase_raw['Gross Total'].isnull() & purchase_raw['Date'].notnull()]

print(f"Sales rows with valid Date but null Gross Total:    {len(sales_null_gt)}")
if len(sales_null_gt) > 0:
    display(sales_null_gt[['Date', 'Particulars', 'Buyer', 'Voucher No.', 'Gross Total', 'source_file']])

print(f"\nPurchase rows with valid Date but null Gross Total: {len(purchase_null_gt)}")
if len(purchase_null_gt) > 0:
    display(purchase_null_gt[['Date', 'Particulars', 'Supplier', 'Voucher No.', 'Gross Total', 'source_file']])

In [ ]:
sales_valid = sales_raw.dropna(subset=['Date', 'Gross Total'])
purchase_valid = purchase_raw.dropna(subset=['Date', 'Gross Total'])

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# Sales — raw
axes[0, 0].hist(sales_valid['Gross Total'], bins=50, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Sales — Gross Total (raw scale)')
axes[0, 0].set_xlabel('Gross Total (INR)')
axes[0, 0].set_ylabel('Invoice count')

# Sales — log
axes[0, 1].hist(sales_valid['Gross Total'], bins=50, color='steelblue', edgecolor='white')
axes[0, 1].set_xscale('log')
axes[0, 1].set_title('Sales — Gross Total (log scale)')
axes[0, 1].set_xlabel('Gross Total (INR, log)')

# Purchases — raw
axes[1, 0].hist(purchase_valid['Gross Total'], bins=50, color='coral', edgecolor='white')
axes[1, 0].set_title('Purchases — Gross Total (raw scale)')
axes[1, 0].set_xlabel('Gross Total (INR)')
axes[1, 0].set_ylabel('Invoice count')

# Purchases — log
axes[1, 1].hist(purchase_valid['Gross Total'], bins=50, color='coral', edgecolor='white')
axes[1, 1].set_xscale('log')
axes[1, 1].set_title('Purchases — Gross Total (log scale)')
axes[1, 1].set_xlabel('Gross Total (INR, log)')

fig.suptitle('Invoice value distribution — raw vs log', fontsize=13)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/day2_01_gross_total_distribution.png", bbox_inches='tight')
plt.show()

## Anonymisation layer

Apply a deterministic, salted-hash pseudonymisation to `Buyer`, `Supplier`, 
and `GSTIN/UIN`. The reversal mapping is written to `data/private/` and 
excluded from Git. All downstream cleaning, plotting, and modelling use 
pseudonymised data only.

In [ ]:
import hashlib
import json
import os

PRIVATE_DIR = "../data/private"
os.makedirs(PRIVATE_DIR, exist_ok=True)

# A salt makes the hashes non-guessable without the salt itself.
# Change this once, then never change it — otherwise pseudonyms won't match.
SALT = "crg-cashflow-dissertation-2026"

def build_pseudonym_map(values, prefix, salt=SALT):
    """
    Build a deterministic mapping from unique real values to salted-hash pseudonyms.

    Same real value always maps to the same pseudonym — so time-series
    relationships (repeat buyers, recurring suppliers) are preserved.
    """
    unique_values = sorted(set(v for v in values if pd.notnull(v)))
    mapping = {}
    for v in unique_values:
        digest = hashlib.sha256(f"{salt}::{v}".encode()).hexdigest()[:8]
        mapping[v] = f"{prefix}_{digest}"
    return mapping


def apply_pseudonymisation(df, column, prefix, mapping_store):
    """Replace a column with pseudonyms; capture the mapping for audit."""
    mapping = build_pseudonym_map(df[column], prefix)
    df[column] = df[column].map(mapping)
    mapping_store[column] = mapping
    return df


# Build mappings across BOTH DataFrames so the same real entity gets the same
# pseudonym in sales and purchases (a supplier may also appear as a buyer, etc.)
buyer_supplier_universe = pd.concat([
    sales_raw['Buyer'].dropna(),
    purchase_raw['Supplier'].dropna()
]).unique()

gstin_universe = pd.concat([
    sales_raw['GSTIN/UIN'].dropna(),
    purchase_raw['GSTIN/UIN'].dropna()
]).unique()

party_map = build_pseudonym_map(buyer_supplier_universe, "PARTY")
gstin_map = build_pseudonym_map(gstin_universe, "GSTIN")

# Apply to both DataFrames
sales_raw['Buyer']         = sales_raw['Buyer'].map(party_map)
sales_raw['GSTIN/UIN']     = sales_raw['GSTIN/UIN'].map(gstin_map)
purchase_raw['Supplier']   = purchase_raw['Supplier'].map(party_map)
purchase_raw['GSTIN/UIN'] = purchase_raw['GSTIN/UIN'].map(gstin_map)

# Save mappings to the private folder (gitignored)
with open(f"{PRIVATE_DIR}/party_mapping.json", "w") as f:
    json.dump(party_map, f, indent=2)
with open(f"{PRIVATE_DIR}/gstin_mapping.json", "w") as f:
    json.dump(gstin_map, f, indent=2)

print(f"Pseudonymised {len(party_map)} unique parties (buyers + suppliers).")
print(f"Pseudonymised {len(gstin_map)} unique GSTINs.")
print(f"Mappings stored (privately) in: {PRIVATE_DIR}/")

In [ ]:
# Drop identifying free-text columns from the working DataFrames.
# We keep the raw parquets on disk in case we need to revisit these fields
# under supervisor guidance; the working DataFrames stay pseudonymised.
free_text_cols_sales    = [c for c in ['Narration'] if c in sales_raw.columns]
free_text_cols_purchase = [c for c in ['Narration', 'Supplier Invoice No.'] 
                           if c in purchase_raw.columns]

sales_raw    = sales_raw.drop(columns=free_text_cols_sales)
purchase_raw = purchase_raw.drop(columns=free_text_cols_purchase)

print(f"Dropped from sales:    {free_text_cols_sales}")
print(f"Dropped from purchase: {free_text_cols_purchase}")

In [ ]:
print("Verification — sample of pseudonymised sales:")
display(sales_raw[['Date', 'Buyer', 'GSTIN/UIN', 'Gross Total']].head(5))

print("\nVerification — sample of pseudonymised purchases:")
display(purchase_raw[['Date', 'Supplier', 'GSTIN/UIN', 'Gross Total']].head(5))

# Overwrite the raw parquets with pseudonymised versions.
# From this point forward, the working data on disk is safe to view/share.
sales_to_save    = sales_raw.copy()
purchase_to_save = purchase_raw.copy()
sales_to_save['Voucher No.']    = sales_to_save['Voucher No.'].astype(str)
purchase_to_save['Voucher No.'] = purchase_to_save['Voucher No.'].astype(str)

sales_to_save.to_parquet("../data/processed/sales_raw.parquet", index=False)
purchase_to_save.to_parquet("../data/processed/purchase_raw.parquet", index=False)

print("\nPseudonymised Parquet files saved.")

## Cleaning and deduplication

Consolidates all Day 3 cleaning decisions into a single audited cell:
(1) drop Grand Total summary rows, (2) drop rows with unusable Date or Gross Total,
(3) detect and remove cross-file duplicates, (4) drop 100%-empty columns,
(5) tag transaction type, (6) save cleaned Parquet files.
Every decision is justified in-code and reported in the summary at the end.

In [ ]:
# =============================================================================
# BLOCK 2 — Consolidated cleaning and deduplication
# =============================================================================
# Design: apply cleaning to COPIES so the raw DataFrames stay untouched in memory,
# and every step reports what it removed for the audit trail.

def clean_invoice_data(df, dataset_name, party_column):
    """
    Apply a fixed sequence of cleaning steps to an invoice DataFrame.
    Returns cleaned DataFrame + a cleaning report dict.
    """
    report = {"dataset": dataset_name, "starting_rows": len(df)}
    df = df.copy()

    # Step 1 — Drop Grand Total summary rows (verified as duplicates of transaction sums).
    grand_totals = df['Particulars'] == 'Grand Total'
    report["grand_total_rows_removed"] = int(grand_totals.sum())
    df = df.loc[~grand_totals].copy()

    # Step 2 — Drop rows with unparseable Date. Without a date, the row cannot
    # contribute to any time-indexed target.
    null_dates = df['Date'].isnull()
    report["null_date_rows_removed"] = int(null_dates.sum())
    df = df.loc[~null_dates].copy()

    # Step 3 — Drop rows with null Gross Total. Cannot aggregate an unknown value.
    null_gt = df['Gross Total'].isnull()
    report["null_gross_total_rows_removed"] = int(null_gt.sum())
    df = df.loc[~null_gt].copy()

    # Step 4 — Drop columns that are 100% null across the whole dataset.
    # These carry no signal and clutter downstream schemas.
    fully_null_cols = [c for c in df.columns if df[c].isnull().all()]
    report["fully_null_columns_dropped"] = fully_null_cols
    df = df.drop(columns=fully_null_cols)

    # Step 5 — Cross-file duplicate detection.
    # Same Date + same party + same Voucher No. + same Gross Total => duplicate.
    dedup_keys = ['Date', party_column, 'Voucher No.', 'Gross Total']
    duplicates_mask = df.duplicated(subset=dedup_keys, keep='first')
    report["duplicates_removed"] = int(duplicates_mask.sum())
    df = df.loc[~duplicates_mask].copy()

    # Step 6 — Tag transaction type; unifies both DataFrames for later joining.
    df['transaction_type'] = dataset_name

    # Step 7 — Sort by date (deterministic ordering, easier debugging).
    df = df.sort_values('Date').reset_index(drop=True)

    report["final_rows"] = len(df)
    report["rows_retained_pct"] = round(100 * len(df) / report["starting_rows"], 2)
    return df, report


sales_clean, sales_report       = clean_invoice_data(sales_raw,    "sales",    "Buyer")
purchase_clean, purchase_report = clean_invoice_data(purchase_raw, "purchase", "Supplier")

# --- Audit summary ------------------------------------------------------------
def print_report(report):
    print(f"\nDataset: {report['dataset']}")
    print(f"  Starting rows:                    {report['starting_rows']:,}")
    print(f"  Grand Total rows removed:         {report['grand_total_rows_removed']}")
    print(f"  Null Date rows removed:           {report['null_date_rows_removed']}")
    print(f"  Null Gross Total rows removed:    {report['null_gross_total_rows_removed']}")
    print(f"  Duplicates removed:               {report['duplicates_removed']}")
    print(f"  Fully-null columns dropped:       {len(report['fully_null_columns_dropped'])} "
          f"{report['fully_null_columns_dropped']}")
    print(f"  Final rows:                       {report['final_rows']:,}")
    print(f"  Retained:                         {report['rows_retained_pct']:.2f}%")

print("=" * 60)
print("CLEANING AUDIT SUMMARY")
print("=" * 60)
print_report(sales_report)
print_report(purchase_report)

# --- Save cleaned Parquet -----------------------------------------------------
sales_to_save    = sales_clean.copy()
purchase_to_save = purchase_clean.copy()
sales_to_save['Voucher No.']    = sales_to_save['Voucher No.'].astype(str)
purchase_to_save['Voucher No.'] = purchase_to_save['Voucher No.'].astype(str)

sales_to_save.to_parquet("../data/processed/sales_clean.parquet",     index=False)
purchase_to_save.to_parquet("../data/processed/purchase_clean.parquet", index=False)

print("\nCleaned Parquet files saved to ../data/processed/")

## Weekly cashflow target construction

Aggregate cleaned sales and purchases to weekly resolution (week ending Sunday),
compute weekly net cashflow, and save as the primary modelling target. This is
the concrete link between raw invoice data (Days 1–3) and downstream modelling
(Phase 3 augmentation and forecasting).

In [ ]:
# Aggregate sales
sales_weekly = (
    sales_clean
    .set_index('Date')['Gross Total']
    .resample('W').sum()
    .rename('sales')
)

# Aggregate purchases
purchase_weekly = (
    purchase_clean
    .set_index('Date')['Gross Total']
    .resample('W').sum()
    .rename('purchases')
)

# Join into a single frame, forcing shared weekly index
cashflow = pd.concat([sales_weekly, purchase_weekly], axis=1).fillna(0)

# Compute net cashflow — the primary target variable
cashflow['net_cashflow'] = cashflow['sales'] - cashflow['purchases']

# ---- Summary --------------------------------------------------------------
print("=" * 60)
print("WEEKLY CASHFLOW TARGET — SUMMARY")
print("=" * 60)
print(f"Weeks in series:              {len(cashflow):,}")
print(f"Date range:                   {cashflow.index.min().date()} → "
      f"{cashflow.index.max().date()}")
print(f"Weeks with zero sales:        {(cashflow['sales'] == 0).sum()}")
print(f"Weeks with zero purchases:    {(cashflow['purchases'] == 0).sum()}")
print(f"Weeks with negative net:      {(cashflow['net_cashflow'] < 0).sum()} "
      f"({100 * (cashflow['net_cashflow'] < 0).mean():.1f}%)")

print("\nDescriptive statistics — weekly net cashflow (INR):")
print(cashflow['net_cashflow'].describe().apply('{:,.2f}'.format))

# ---- Save target ----------------------------------------------------------
cashflow.to_parquet("../data/processed/weekly_cashflow.parquet")
print("\nSaved: ../data/processed/weekly_cashflow.parquet")

In [ ]:
# One supervisor-ready figure summarising the target variable
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

axes[0].plot(cashflow.index, cashflow['sales'],
             color='steelblue', linewidth=1.1)
axes[0].set_ylabel('Weekly sales (INR)')
axes[0].set_title('Weekly sales inflow')

axes[1].plot(cashflow.index, cashflow['purchases'],
             color='coral', linewidth=1.1)
axes[1].set_ylabel('Weekly purchases (INR)')
axes[1].set_title('Weekly purchase outflow')

axes[2].plot(cashflow.index, cashflow['net_cashflow'],
             color='darkgreen', linewidth=1.1)
axes[2].axhline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.6)
axes[2].fill_between(cashflow.index, cashflow['net_cashflow'], 0,
                     where=(cashflow['net_cashflow'] >= 0),
                     color='green', alpha=0.25, label='Surplus week')
axes[2].fill_between(cashflow.index, cashflow['net_cashflow'], 0,
                     where=(cashflow['net_cashflow'] < 0),
                     color='red', alpha=0.25, label='Deficit week')
axes[2].set_ylabel('Weekly net cashflow (INR)')
axes[2].set_title('Weekly net cashflow (target variable)')
axes[2].legend(loc='upper left')
axes[2].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=45, ha='right')

fig.suptitle('SME weekly cashflow across three financial years '
             '(anonymised)', fontsize=13)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/day4_weekly_cashflow_target.png",
            bbox_inches='tight')
plt.show()

In [ ]:
# Check if today's work variables still exist in kernel memory
variables_to_check = [
    'buyer_terms',
    'supplier_terms',
    'supplier_terms_dedup',
    'buyer_terms_dedup',
    'sales_with_terms',
    'purchase_with_terms',
    'cashflow_projected',
    'cashflow_projected_v2',
    'per_buyer_threshold',
    'DEFAULT_TERM',
    'parse_terms',
    'project_to_payment_day',
]

for var in variables_to_check:
    try:
        obj = eval(var)
        if hasattr(obj, 'shape'):
            print(f"✓ {var:35} exists — shape {obj.shape}")
        elif callable(obj):
            print(f"✓ {var:35} exists — function")
        else:
            print(f"✓ {var:35} exists — {type(obj).__name__}")
    except NameError:
        print(f"✗ {var:35} — NOT in memory")

In [ ]:
# Safety snapshot of every variable currently in kernel memory
import os
os.makedirs("../data/processed", exist_ok=True)

# Force Voucher No. to string to avoid Arrow type errors (same fix as before)
for df_name in ['sales_with_terms', 'purchase_with_terms']:
    df = eval(df_name)
    if 'Voucher No.' in df.columns:
        df['Voucher No.'] = df['Voucher No.'].astype(str)

# Save every DataFrame
sales_with_terms.to_parquet("../data/processed/sales_with_terms_RECOVERED.parquet")
purchase_with_terms.to_parquet("../data/processed/purchase_with_terms_RECOVERED.parquet")
cashflow_projected.to_parquet("../data/processed/cashflow_projected_RECOVERED.parquet")
cashflow_projected_v2.to_parquet("../data/processed/cashflow_projected_v2_RECOVERED.parquet")

buyer_terms.to_parquet("../data/processed/buyer_terms_RECOVERED.parquet")
supplier_terms.to_parquet("../data/processed/supplier_terms_RECOVERED.parquet")
buyer_terms_dedup.to_parquet("../data/processed/buyer_terms_dedup_RECOVERED.parquet")
supplier_terms_dedup.to_parquet("../data/processed/supplier_terms_dedup_RECOVERED.parquet")
per_buyer_threshold.to_frame().to_parquet("../data/processed/per_buyer_threshold_RECOVERED.parquet")

print("All variables saved to disk. Data is now recoverable even if kernel dies.")

In [ ]:
# Verify notebook 02 has the variables from its earlier cells
required_vars = ['sales_clean', 'purchase_clean', 'features', 'cashflow', 'BASE_SERIES']

for var in required_vars:
    try:
        obj = eval(var)
        if hasattr(obj, 'shape'):
            print(f"✓ {var:15} exists — shape {obj.shape}")
        else:
            print(f"✓ {var:15} exists — {type(obj).__name__}")
    except NameError:
        print(f"✗ {var:15} — NOT in memory")

In [ ]:
import os
processed_files = sorted(os.listdir("../data/processed"))
print("Files in data/processed/:")
for f in processed_files:
    print(f"  {f}")

In [ ]:
import pandas as pd
print(pd.read_parquet("../data/processed/weekly_cashflow_projected.parquet").shape)

In [ ]:
import pandas as pd
for f in ['weekly_cashflow.parquet',
          'weekly_cashflow_projected.parquet',
          'cashflow_projected_RECOVERED.parquet',
          'cashflow_projected_v2_RECOVERED.parquet']:
    try:
        print(f, pd.read_parquet(f"../data/processed/{f}").shape)
    except Exception as e:
        print(f, "->", type(e).__name__)